# Data Cleaning with Pandas

In this notebook we'll go through a few basic data cleaning steps that should be performed on all new datasets where necessary.

We'll go through the process with both the `orders` and `orderlines` datasets. You can then practice these skills by cleaning the `products` dataset yourself

In [1]:
import pandas as pd

In [2]:
# orders.csv
url = "https://drive.google.com/file/d/1Vu0q91qZw6lqhIqbjoXYvYAQTmVHh6uZ/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orders = pd.read_csv(path)

# orderlines.csv
url = "https://drive.google.com/file/d/1FYhN_2AzTBFuWcfHaRuKcuCE6CWXsWtG/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orderlines = pd.read_csv(path)

Before we begin, let's create a copy of the `orders` and `orderlines` DataFrames. This way we are sure any of our changes won't affect the original DataFrames

In [3]:
orders_df = orders.copy()

In [4]:
orderlines_df = orderlines.copy()

One of the best ways to begin data cleaning is by exploring using `.info()`. This will tell us:
* The shape of the DataFrame
* The names of the columns
* If there are any missing values
* The datatypes of the columns

By exploring the missing values and correcting any incorrect datatypes, we often come across inconsistencies in our data.

Beyond this, we should also have a **check for any duplicate rows**.

Let's first deal with the duplicates, as it's nice and easy, then we'll explore what `.info()` has to tell us.

## 1.&nbsp; Duplicates
We can check for duplicates using the pandas [.duplicated()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html) method.

We can then delete these rows, if we wish, using [.drop_duplicates()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html)

In [5]:
# orders_df
orders_df.duplicated().sum()

np.int64(0)

In [6]:
# orderlines_df
orderlines_df.duplicated().sum()

np.int64(0)

We have no duplicate rows in either DataFrame. Easy, there is no problem to solve. Normally though, if there were some duplicates, we'd drop the extra rows.

# 2.&nbsp; `.info()`

In [7]:
orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      226909 non-null  int64  
 1   created_date  226909 non-null  object 
 2   total_paid    226904 non-null  float64
 3   state         226909 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 6.9+ MB


* `total_paid` has 5 missing values
* `created_date` should become datetime datatype

In [8]:
orderlines_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   id                293983 non-null  int64 
 1   id_order          293983 non-null  int64 
 2   product_id        293983 non-null  int64 
 3   product_quantity  293983 non-null  int64 
 4   sku               293983 non-null  object
 5   unit_price        293983 non-null  object
 6   date              293983 non-null  object
dtypes: int64(4), object(3)
memory usage: 15.7+ MB


* `date` should be a datetime datatype
* `unit_price` should be a float datatype

## 3.&nbsp; Missing values

### 3.1.&nbsp; Orders
* `total_paid` has 5 missing values

In [15]:
num_missing = orders_df['total_paid'].isna().sum()
total_rows = orders_df.shape[0]
percent_missing = (100*num_missing/total_rows)
print(f"5 missing values represents {percent_missing:.5f}% of the rows in our DataFrame")

5 missing values represents 0.00000% of the rows in our DataFrame


> A quick way to find out a percentage if you don't need to print out a sentence for yourself/students/colleagues is `.value_counts(normalize=True)`

In [16]:
orders_df['total_paid'].isna().value_counts(normalize=True)

,proportion
total_paid,
False,1.0


As there is such a tiny amount of missing values, we will simply delete these rows, as we have enough data without them.

In [17]:
orders_df = orders_df.dropna(axis=0)

Should you have a significant number of missing values in the future, you have a choice:
+ you can impute the values
+ you can take the values from other DataFrames if they are redundantly stored
+ you can delete the rows or columns
+ or any number of other creative solutions

Please, always consider how much time you have on your project, and what impact your method of choice will have on your final assesment.

### 3.2.&nbsp; Orderlines
There are no missing values in `orderlines_df`

## 4.&nbsp; Datatypes

### 4.1.&nbsp; Orders
* `created_date` should become datetime datatype

In [12]:
orders_df["created_date"] = pd.to_datetime(orders_df["created_date"])

### 4.1.&nbsp; Orderlines
* `date` should be a datetime datatype
* `unit_price` should be a float datatype

#### 4.1.1.&nbsp; `date`

In [13]:
orderlines_df["date"] = pd.to_datetime(orderlines_df["date"])

#### 4.1.2.&nbsp;`unit_price`

In [18]:
orderlines_df["unit_price"] = pd.to_numeric(orderlines_df["unit_price"])

ValueError: Unable to parse string "1.137.99" at position 6

As you can see when we try to convert `unit_price` to a numerical datatype, we receive a `ValueError` telling us that pandas doesn't understand the number `1.137.99`. This is probably because numbers cannot have multiple decimal points. Let's see if there are any other numbers like this:

> `.` is a wildcard in regex, we need the `\` as an escape

In [19]:
# Count the number of decimal points in the unit_price
orderlines_df['unit_price'].str.count(r"\.").value_counts()

,count
unit_price,
1,257814
2,36169


Looks like over 36000 rows in `orderlines` are affected by this problem. Let's work out how much that is as a percentage of our total data.

In [20]:
# Count the rows with more than one `.`
mult_decimal_rows = (orderlines_df['unit_price'].str.count(r"\.")>1).sum()

# Find the percentage of corrupted rows
percent_corrupted = (100 * mult_decimal_rows / orderlines_df.shape[0])
print(f"{percent_corrupted:.2f}% of the rows in our DataFrame have multiple decimal points in the unit_price")

12.30% of the rows in our DataFrame have multiple decimal points in the unit_price


This is a bit of a tricky decision as 12.3% is a significant amount of our data... and we might even end up losing a larger portion of our data than this too. For the moment we will delete the rows as we only have 2 weeks for this project and I'd like some quick, accurate results to show. If we have time at the end, we can come back and investigate this problem further, maybe there's a solution?

Each row of `orderlines` represents a product in an order. For example, if order number 175 contained 3 seperate products, then order 175 would have 3 rows in `orderlines`, one row for each of the products. If 2 of those products have 'normal' prices (14.99, 15.85) and 1 has a price with 2 decimal points (1.137.99), we need to remove the whole order and not just the affected row. If we only remove the row with 2 decimal places then any later analysis about products and prices could be misleading.

We therefore need to find the order numbers associated with the rows that have 2 decimal points, and then remove all the associated rows.

In [21]:
# Boolean mask to find the orders that contain a price with multiple decimal points
multiple_decimal_mask = orderlines_df['unit_price'].str.count(r"\.") > 1

# Apply the boolean mask to the orderlines DataFrame. This way we can find the order_id of all the affected orders.
corrupted_order_ids = orderlines_df.loc[multiple_decimal_mask, "id_order"]

# Keep only the rows that do not have multiple decimal points
orderlines_df = orderlines_df.loc[~orderlines_df['id_order'].isin(corrupted_order_ids)]

In [22]:
orderlines_df.shape[0]

216250

We still have 216250 rows in orderlines to work with. This should be more than enough for our evaluation.

Now that all of the 2 decimal point prices have been removed, let's try again to convert the column `unit_price` to the correct datatype.

In [23]:
orderlines_df["unit_price"] = pd.to_numeric(orderlines_df["unit_price"])

It worked perfectly

# Clean the `products` DataFrame
clean the products DataFrame. You don't have to copy exactly what we did. Think about the consequences of your actions, sometimes it is ok to delete rows, other times you may wish to come up with more creative solutions.

In [24]:
# products.csv
url = "https://drive.google.com/file/d/1afxwDXfl-7cQ_qLwyDitfcCx3u7WMvkU/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
products = pd.read_csv(path)

In [25]:
products_df = products.copy()

In [26]:
products_df.sample(5)

,sku,name,desc,price,promo_price,in_stock,type
13919,APP1863,"Apple MacBook Pro 13 ""with Touch Bar 33GHz Cor...",New MacBook Pro 13 inch Touch Bar 33 GHz Core ...,2559,24.375.849,0,2158
4132,APP1389,"Apple iMac 27 ""Core i5 3.3GHz Retina 5K | 8GB ...",IMac desktop computer 27 inch 8GB RAM 512GB Re...,3169,30.175.839,0,"5,74E+15"
14774,PAC1710,Pack QNAP TS-451A 2GB NAS server 40TB (4x10TB)...,NAS with 2GB of RAM and 40TB (4x10TB) Seagate ...,2046.95,15.173.678,1,12175397
4861,PAC1041,"Apple iMac 27 ""Core i7 Retina 5K 4GHz | 16GB |...",IMac desktop computer 27 inch 5K Retina i7 4GH...,3169,28.739.896,0,"5,74E+15"
5766,PAC1057,"Apple iMac 27 ""Core i7 Retina 5K 4GHz | 16GB |...",IMac desktop computer 27 inch 5K Retina 4GHz i...,3589,32.739.902,0,"5,74E+15"


### Look for Duplicates

In [27]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19326 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   sku          19326 non-null  object
 1   name         19326 non-null  object
 2   desc         19319 non-null  object
 3   price        19280 non-null  object
 4   promo_price  19326 non-null  object
 5   in_stock     19326 non-null  int64 
 6   type         19276 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.0+ MB


In [28]:
products_df.duplicated().sum()

np.int64(8746)

In [29]:
products_df = products_df.drop_duplicates()

### Look for Missing values


In [ ]:
products_df = products_df.drop_duplicates()

### SKU specific Duplicate Logic

In [ ]:
products_df["sku"].duplicated().sum()

NameError: name 'products_df' is not defined

Why is there a duplicate in SKU?

#### Investigation

In [ ]:
products_df.loc[products_df["sku"].duplicated(keep=False)]
# keep = False marks every value that appears more than once as True

In [ ]:
products_df.loc[products_df["sku"] == "APP1197"]

In [ ]:
products_df.drop_duplicates(subset='sku', keep="first", inplace=True)

## `.info()`

In [ ]:
products_df.info()

NameError: name 'products_df' is not defined

### Missing values
We can see from `.info()` above that we have missing values in `desc` and `price`

#### `desc`

In [ ]:
products_df["desc"].isna().sum()

7 is a very small number to have missing, let's have a closer look

In [ ]:
products_df.loc[products_df['desc'].isna(), :]

We have 2 choices here:
* We can quickly and easily remove these rows.
* Or, alternatively, the products names here are quite descriptive, so I'm tempted to just copy them to the description column, so that there is a description if we later want utilise this column. I wouldn't recommend this if this DataFrame was the source of truth for our website. But this is not the case here, and we're not faking any information (guessing a price or so), so I'm happy with this option

In [ ]:
products_df.loc[products_df['desc'].isna(), 'desc']

In [ ]:
products_df.loc[products_df['desc'].isna(), 'name']

In [ ]:
products_df.loc[products_df['desc'].isna(), 'desc'] = products_df.loc[products_df['desc'].isna(), 'name']

In [ ]:
products_df.loc[products_df['desc'].isna(), :]

Did you also notice above that we have the dreaded two decimal point problem in both the `price` and `promo_price` columns? We can also see prices with 3 decimal places, prices should have 2 decimal places: this gives us more cause for concern

#### `price`

In [ ]:
products_df.price.isna().sum()

np.int64(45)

In [ ]:
print(f"The missing values in price are {(products_df.price.isna().value_counts(normalize=True).iloc[1] * 100).round(2)}% of all rows in the DataFrame")

The missing values in price are 0.43% of all rows in the DataFrame


Let's simply delete these rows to ensure that we can trust the numbers in our final DataFrame. Afterall, the price is very important when investigating discounts.

Option 1: `.loc`

In [ ]:
products_df.loc[~products['price'].isna()]

,sku,name,desc,price,promo_price,in_stock,type
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,499.899,1,8696
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59,589.996,0,13855401
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59,569.898,0,1387
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25,229.997,0,1230
4,KIN0007,Mac Memory Kingston 2GB 667MHz DDR2 SO-DIMM,2GB RAM Mac mini and iMac (2006/07) MacBook Pr...,34.99,31.99,1,1364
...,...,...,...,...,...,...,...
19321,BEL0376,Belkin Travel Support Apple Watch Black,compact and portable stand vertically or horiz...,29.99,269.903,1,12282
19322,THU0060,"Enroute Thule 14L Backpack MacBook 13 ""Black",Backpack with capacity of 14 liter compartment...,69.95,649.903,1,1392
19323,THU0061,"Enroute Thule 14L Backpack MacBook 13 ""Blue",Backpack with capacity of 14 liter compartment...,69.95,649.903,1,1392
19324,THU0062,"Enroute Thule 14L Backpack MacBook 13 ""Red",Backpack with capacity of 14 liter compartment...,69.95,649.903,0,1392


In [35]:
products_df = products_df.loc[~products['price'].isna()]

Option 2: `.dropna()`

In [36]:
products_df = products_df.dropna(subset=['price'])

#### `type`

In [37]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9992 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   sku          9992 non-null   object
 1   name         9992 non-null   object
 2   desc         9986 non-null   object
 3   price        9992 non-null   object
 4   promo_price  9992 non-null   object
 5   in_stock     9992 non-null   int64 
 6   type         9946 non-null   object
dtypes: int64(1), object(6)
memory usage: 624.5+ KB


Type isn’t an essential piece of data for the analysis and is therefore allowed to carry missing values.
The only place it comes in later is as an optional route to category creation, where someone might still choose to drop the rows with missing values, however one can still use name and desc to categorize those rows.


### Data types

We saw from looking at the output of `.info()` that both `price` and `promo_price` have been stored as objects and not as a numerical datatypes. We also saw while solving other problems that both columns have some prices with 3 decimal places and others with 2 decimal points - the latter will prevent us from converting the datatype to numerical, so first we must investigate and solve these problems.

#### `price`

First, let's see how many values are affected by the 2-decimal-dot problems or 3 decimal places.

In [34]:
products_df["price"]

,price
0,59.99
1,59
2,59
3,25
4,34.99
...,...
19321,29.99
19322,69.95
19323,69.95
19324,69.95


In [38]:
products_df.loc[
    (products_df.price.str.contains(r"\d+\.\d+\.\d+"))|(products_df.price.str.contains(r"\d+\.\d{3,}")), :].sample(10)

ValueError: a must be greater than 0 unless no samples are taken

In [39]:
price_problems_number = products_df.loc[
    (products_df.price.str.contains(r"\d+\.\d+\.\d+"))|(products_df.price.str.contains(r"\d+\.\d{3,}")), :].shape[0]
price_problems_number

0

In [40]:
print(f"The column price has in total {price_problems_number} wrong values. This is {round(((price_problems_number / products_df.shape[0]) * 100), 2)}% of the rows of the DataFrame")

The column price has in total 0 wrong values. This is 0.0% of the rows of the DataFrame


5.15% is a reasonable amount of our data. However, the price column will be important to understanding discounts, so I'd like it to be very trustworthy as we are basing business decisions on it. Therefore, we'll delete these rows

In [33]:
products_df = products_df.loc[~((products_df.price.str.contains(r"\d+\.\d+\.\d+"))|(products_df.price.str.contains(r"\d+\.\d{3,}"))), :]

In [41]:
products_df.sample(10)

,sku,name,desc,price,promo_price,in_stock,type
16981,BNQ0056,"PD3200U Monitor Benq 32 ""4K UHD professional d...",32 inch monitor with 100% sRGB 4K height adjus...,773,7.729.988,0,1296
16888,APP2018,Apple Mac Pro 35GHz 6 cores | 16GB RAM | 256GB...,New Mac Pro with 16GB of RAM and two 6-core GP...,3455.59,31.990.041,0,21632158
14711,OTT0153-A,Open - Otterbox Clearly Protected Case iPhone ...,TPU flexible transparent cover light a piece f...,19.99,90.322,0,1298
13727,PAR0068,Mando Parrot FLYPAD,Mando compatible with minidrones includes supp...,39,369.897,0,11905404
1364,APP0933,Apple Mac mini Core i7 3hz | 16GB RAM | 1TB HDD,PC Mac mini Core i7 3GHz 16GB 1TB (MGEN2YP / A).,1339,1.272.004,0,1282
2539,FCM0032,Mac memory DIMM DDR3 2GB 1066MHz FCM,2GB Memory Mac Pro (March 2009 julio2010).,19.99,149.895,0,1364
18766,MOS0205-A,Open - Moshi iGlaze XT Case iPhone 7/8 Transpa...,Reconditioned sleeve transparent ultra thin wi...,30,170.023,0,11865403
17949,STA0037,Startech USB-C HDMI Adapter Black,Adapter with USB-C reversible HDMI connection ...,39.99,299.899,0,12585395
17233,AP20285,Like new - Apple iPod Shuffle 2GB Blue,Music player iPod Shuffle 2GB refitted.,55,42.811,0,11821715
17687,APP2541,Apple iPhone leather case Leather Case 8 Plus ...,ultrathin leather case and microfibre premium ...,59,51,0,11865403


To complete our task, let's convert the column to a numeric datatype

In [42]:
products_df["price"] = pd.to_numeric(products_df["price"])

In [31]:
products_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10580 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   sku          10580 non-null  object
 1   name         10580 non-null  object
 2   desc         10573 non-null  object
 3   price        10534 non-null  object
 4   promo_price  10580 non-null  object
 5   in_stock     10580 non-null  int64 
 6   type         10530 non-null  object
dtypes: int64(1), object(6)
memory usage: 661.2+ KB


#### `promo_price`

Again, let's begin by seeing how many values are affected by the 2-decimal-dots problem, or the 3 decimal-places problem

In [30]:
promo_problems_number = products_df.loc[(products_df.promo_price.str.contains(r"\d+\.\d+\.\d+"))|(products_df.promo_price.str.contains(r"\d+\.\d{3,}")), :].shape[0]
promo_problems_number

9734

In [43]:
print(f"The column promo_price has in total {promo_problems_number} wrong values. This is {round(((promo_problems_number / products_df.shape[0]) * 100), 2)}% of the rows of the DataFrame")

The column promo_price has in total 9734 wrong values. This is 97.42% of the rows of the DataFrame


WOW!!! That's a lot of wrong data. Let's have a quick investigate to check that's correct. We'll make a DataFrame by copy-pasting the code we used above and then look at a large sample to check that all the numbers in the `promo_price` column really have either 2 decimal points or 3 decimal places.

In [44]:
promo_price_df = products_df.loc[(products_df.promo_price.str.contains(r"\d+\.\d+\.\d+"))|(products_df.promo_price.str.contains(r"\d+\.\d{3,}")), :]
promo_price_df.sample(10)

,sku,name,desc,price,promo_price,in_stock,type
17031,OWC0230,OWC USB-Dock C-10 80W power ports Oro,Aluminum Hub with 10 different ports include 2...,217.99,1.799.899,1,12995397
12497,SPE0170,Speck CandyShell iPhone 6 / 6S Black,Ultra resistant transparent housing black and ...,24.95,179.915,0,11865403
17272,TUC0327,Tucano tugo Medium Backpack 20L MacBook Pro 13...,Backpack 20 liter single compartment handles a...,59.90,299.899,0,1392
17873,PAC2360,Synology DS218 + NAS Server | 2GB RAM | 4TB (2...,NAS storage server integrated with special foc...,564.97,4.911.789,0,12175397
2508,IFX0005-A,Open - iFixit Macro Bit Kit repair steel parts 97,97 Kit of tools for repair iPhone iPad Mac and...,35.99,273.656,0,1298
13316,APP1750,Apple iPad Air 2 Wi-Fi 32GB Dorado,New iPad Air 2 Wi-Fi 32GB (MNV72TY / A).,429.00,432.811,0,42931714
12109,LAC0182,LaCie Porsche Design Mobile Hard Drive 4TB USB...,4TB External Hard Drive USB-C and USB 3.0 conn...,199.99,1.359.943,0,11935397
12086,STM0100,"Grace Deluxe Bag STM MacBook Pro 13 ""Blue / Black",Bag printed with magnetic closure and handles ...,49.99,399.905,0,1392
15175,LEA0001-A,Open - Leap Motion Sensor Controller gestures,Sensor motion controller for Mac,89.99,727.222,0,1298
16113,AP20128,Like new - Adapter Apple Lightning to SD Card ...,Reconditioned adapter connector Lightning to S...,35.00,249.901,0,14365395


So we were correct, over 90% of the data in this column is corrupt. There's no point deleting all of these rows, then we would barely have a products table. Instead, as it's only this column that appears to be very untrustworthy, we will delete the column.

In [45]:
products_cl = products_df.drop(columns=["promo_price"])

In [46]:
products_cl.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9992 entries, 0 to 19325
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   sku       9992 non-null   object 
 1   name      9992 non-null   object 
 2   desc      9986 non-null   object 
 3   price     9992 non-null   float64
 4   in_stock  9992 non-null   int64  
 5   type      9946 non-null   object 
dtypes: float64(1), int64(1), object(4)
memory usage: 546.4+ KB


Obviously, there's now no need to convert `promo_price` to a numerical datatype

Don't forget to download/save your new DataFrames. Also, give them an obvious name, so that you know they are the cleaned version and not the original DataFrame.

In [47]:
from google.colab import files

orders_df.to_csv("orders_cl.csv", index=False)
files.download("orders_cl.csv")

orderlines_df.to_csv("orderlines_cl.csv", index=False)
files.download("orderlines_cl.csv")

products_cl.to_csv("products_cl.csv", index=False)
files.download("products_cl.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>